In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [ ]:

# XOR truth table
X = torch.tensor([
    [0., 0.],
    [0., 1.],
    [1., 0.],
    [1., 1.],
], dtype=torch.float32)

y = torch.tensor([
    [0.],
    [1.],
    [1.],
    [0.],
], dtype=torch.float32)

# we repeat it to make our dataset
X_train = X.repeat(256, 1)
y_train = y.repeat(256, 1)

ds = TensorDataset(X_train, y_train)
dl = DataLoader(ds, batch_size=32, shuffle=True)

X.to(device), y.to(device)


(tensor([[0., 0.],
         [0., 1.],
         [1., 0.],
         [1., 1.]], device='cuda:0'),
 tensor([[0.],
         [1.],
         [1.],
         [0.]], device='cuda:0'))

In [ ]:
# Tiny XOR net: 2 -> 2 -> 1
class TinyXORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 2)
        self.act = nn.Tanh()
        self.fc2 = nn.Linear(2, 1)
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

model = TinyXORNet().to(device)
print("params:", sum(p.numel() for p in model.parameters()))




params: 9


In [ ]:
#training loop
model = TinyXORNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.BCEWithLogitsLoss()

model.train()
for epoch in range(300):
    total = 0.0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = loss_fn(logits, yb)

        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
    if (epoch + 1) % 50 == 0:
        print(f"epoch {epoch+1:3d} | loss {total/len(dl):.6f}")


epoch  50 | loss 0.000811
epoch 100 | loss 0.000204
epoch 150 | loss 0.000075
epoch 200 | loss 0.000031
epoch 250 | loss 0.000014
epoch 300 | loss 0.000006


In [ ]:

model.eval()
with torch.no_grad():
    logits = model(X.to(device))
    probs = torch.sigmoid(logits).cpu()
    preds = (probs >= 0.5).float()



In [ ]:
model.eval()
with torch.no_grad():
    p = torch.sigmoid(model(X.to(device))).cpu()

print("pred :", (p.squeeze() >= 0.5).int().tolist())
print("true :", y.squeeze().int().tolist())

print("\nW1:\n", model.fc1.weight.detach().cpu().numpy())
print("b1:\n", model.fc1.bias.detach().cpu().numpy())
print("\nW2:\n", model.fc2.weight.detach().cpu().numpy())
print("b2:\n", model.fc2.bias.detach().cpu().numpy())


pred : [0, 1, 1, 0]
true : [0, 1, 1, 0]

W1:
 [[ 5.748306  -5.5051546]
 [-5.9620376  6.3590474]]
b1:
 [2.6125793 2.763279 ]

W2:
 [[-12.332632 -12.201472]]
b2:
 [11.756237]


In [ ]:
#part 2: 16-bit PARITY
# =========================================
torch.manual_seed(1)

def make_parity_dataset(n_samples: int, n_bits: int = 16):
    Xb = torch.randint(0, 2, (n_samples, n_bits)).float()
    # parity = 1 if odd number of 1s else 0
    yb = (Xb.sum(dim=1) % 2).unsqueeze(1).float()
    return Xb, yb

n_bits = 16
X_train, y_train = make_parity_dataset(60000, n_bits)
X_val,   y_val   = make_parity_dataset(10000, n_bits)
X_test,  y_test  = make_parity_dataset(10000, n_bits)

train_dl = DataLoader(TensorDataset(X_train, y_train), batch_size=256, shuffle=True)
val_dl   = DataLoader(TensorDataset(X_val, y_val), batch_size=512, shuffle=False)

X_train.shape, y_train.shape


(torch.Size([60000, 16]), torch.Size([60000, 1]))

In [ ]:

class ParityNet(nn.Module):
    def __init__(self, n_bits: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_bits, 256),
            nn.Tanh(),
            nn.Linear(256, 256),
            nn.Tanh(),
            nn.Linear(256, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x)

parity_model = ParityNet(n_bits).to(device)

num_params = sum(p.numel() for p in parity_model.parameters())
print("Number of parameters:", num_params)

opt = torch.optim.Adam(parity_model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()


Number of parameters: 103169


In [ ]:

def accuracy_from_logits(logits, y_true):
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).float()
    return (preds == y_true).float().mean().item()

for epoch in range(300):
    parity_model.train()
    train_loss = 0.0
    train_acc = 0.0

    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        logits = parity_model(xb)
        loss = loss_fn(logits, yb)

        opt.zero_grad()
        loss.backward()
        opt.step()

        train_loss += loss.item()
        train_acc += accuracy_from_logits(logits.detach(), yb)

    parity_model.eval()
    val_loss = 0.0
    val_acc = 0.0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = parity_model(xb)
            val_loss += loss_fn(logits, yb).item()
            val_acc += accuracy_from_logits(logits, yb)

    train_loss /= len(train_dl)
    train_acc  /= len(train_dl)
    val_loss   /= len(val_dl)
    val_acc    /= len(val_dl)

    print(f"epoch {epoch+1:2d} | train loss {train_loss:.4f} acc {train_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")


epoch  1 | train loss 0.6931 acc 0.5074 | val loss 0.6932 acc 0.5034
epoch  2 | train loss 0.6931 acc 0.5065 | val loss 0.6932 acc 0.5026
epoch  3 | train loss 0.6932 acc 0.5047 | val loss 0.6932 acc 0.5033
epoch  4 | train loss 0.6933 acc 0.5065 | val loss 0.6938 acc 0.5016
epoch  5 | train loss 0.6932 acc 0.5086 | val loss 0.6937 acc 0.4976
epoch  6 | train loss 0.6932 acc 0.5047 | val loss 0.6933 acc 0.5023
epoch  7 | train loss 0.6931 acc 0.5088 | val loss 0.6939 acc 0.4947
epoch  8 | train loss 0.6930 acc 0.5078 | val loss 0.6936 acc 0.5042
epoch  9 | train loss 0.6929 acc 0.5095 | val loss 0.6936 acc 0.5009
epoch 10 | train loss 0.6929 acc 0.5072 | val loss 0.6937 acc 0.4995
epoch 11 | train loss 0.6929 acc 0.5086 | val loss 0.6942 acc 0.4978
epoch 12 | train loss 0.6931 acc 0.5086 | val loss 0.6939 acc 0.4962
epoch 13 | train loss 0.6777 acc 0.5420 | val loss 0.5489 acc 0.6489
epoch 14 | train loss 0.2419 acc 0.8724 | val loss 0.0849 acc 0.9619
epoch 15 | train loss 0.0460 acc 0

In [ ]:

parity_model.eval()
with torch.no_grad():
    Xt, yt = X_test.to(device), y_test.to(device)
    test_logits = parity_model(Xt)
    test_acc = accuracy_from_logits(test_logits, yt)

print("Test accuracy:", test_acc)

with torch.no_grad():
    probs = torch.sigmoid(test_logits[:10]).cpu().squeeze(1)
    preds = (probs >= 0.5).float()
print("\nFirst 10 samples (bits -> true -> pred -> prob):")
for i in range(10):
    bits = "".join(str(int(b.item())) for b in X_test[i])
    print(bits, " | true", int(y_test[i].item()), "| pred", int(preds[i].item()), "| p=", float(probs[i]))

print("\nNumber of parameters:", sum(p.numel() for p in parity_model.parameters()))


Test accuracy: 0.9997999668121338

First 10 samples (bits -> true -> pred -> prob):
1100111100000010  | true 1 | pred 1 | p= 0.9999923706054688
0011001000000110  | true 1 | pred 1 | p= 0.9999544620513916
1000011000111111  | true 1 | pred 1 | p= 0.9999985694885254
0100000010001011  | true 1 | pred 1 | p= 0.9999395608901978
0111011100111101  | true 1 | pred 1 | p= 0.9999982118606567
1011001000000100  | true 1 | pred 1 | p= 0.9999982118606567
0100100100100000  | true 0 | pred 0 | p= 7.105928716555354e-07
0111000011011010  | true 0 | pred 0 | p= 3.6141912573839363e-07
0000100001011101  | true 0 | pred 0 | p= 2.4326752736669732e-06
1101010100010010  | true 1 | pred 1 | p= 0.999998927116394

Number of parameters: 103169
